# Libraries

In [ ]:
import yfinance as yf

ModuleNotFoundError: No module named 'yfinance'

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

: 

In [ ]:
import seaborn as sns

In [ ]:
import matplotlib.pyplot as plt 

In [ ]:
import numpy as np

In [ ]:
import gradio as gr

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

## Data Collection

#### Independent variable - USD - INR

In [ ]:
usd_inr = yf.download('USDINR=X', start='2025-01-01', end='2025-12-31', interval='1wk')

In [ ]:
type(usd_inr)

In [ ]:
usd_inr.head()

In [ ]:
usd_inr.info()

In [ ]:
usd_inr.reset_index(inplace=True)

In [ ]:
# Keep only relevant columns
usd_inr = usd_inr[['Date', 'Close']]
usd_inr.columns = ['Date', 'USD_INR']

In [ ]:
usd_inr.head()

#### Dependent Variable y - Gold rate

In [ ]:
gold_data_inr = yf.download('XAU', start='2025-01-01', end='2025-12-31', interval='1wk')

##### Web scraping using BeautifulSoup

In [ ]:
import yfinance as yf
import pandas as pd

# Download gold price (USD per ounce)
gold = yf.download("GC=F", start="2024-01-01", end="2025-12-31")

# Download USD to INR rate
usd = yf.download("USDINR=X", start="2024-01-01", end="2025-12-31")

# Keep only Close column
gold = gold[['Close']]
usd = usd[['Close']]

# Rename columns
gold.columns = ['Gold_USD_per_Ounce']
usd.columns = ['USD_Rate']

# Merge datasets using Date
data = pd.merge(gold, usd, left_index=True, right_index=True)

# Remove missing values
data.dropna(inplace=True)

# Convert Gold USD → INR
data['Gold_INR_per_Ounce'] = data['Gold_USD_per_Ounce'] * data['USD_Rate']

# Convert ounce → gram
data['Gold_Price_per_Gram_INR'] = data['Gold_INR_per_Ounce'] / 31.1035

# Reset index
data.reset_index(inplace=True)

# Convert daily → weekly (every 7 days)
data['Date'] = pd.to_datetime(data['Date'])
data.set_index('Date', inplace=True)

weekly_data = data.resample('7D').first()
weekly_data.reset_index(inplace=True)

# Select required columns
weekly_data = weekly_data[['Date','USD_Rate','Gold_Price_per_Gram_INR']]

print(weekly_data.head())

# Save dataset
weekly_data.to_csv("gold_weekly_dataset.csv", index=False)

print("Dataset saved successfully")

In [ ]:
gold_dataset = pd.read_csv("gold_weekly_dataset.csv") 
gold_dataset.head(150)

In [ ]:
gold_dataset.info()

## Data Analysis

In [ ]:
gold_dataset.head()

In [ ]:
gold_dataset.rename(columns={"USD_Rate":"USD_INR"}, inplace=True)
print(gold_dataset.head())

## EDA

- Handle missing values - No
- Handle Imbalanced dataset - No
- Handle outliers - Seen
- Encode categorical features - No
- Normalization vs Standardisation - standardize

In [ ]:
sns.boxplot(gold_dataset['USD_INR'])

In [ ]:
gold_dataset['USD_INR'].min()

## Model Training

In [ ]:
X = gold_dataset[['USD_INR']]
y = gold_dataset[['Gold_Price_per_Gram_INR']]

In [ ]:
X

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.1,random_state=42)

In [ ]:
X_train.shape, X_test.shape

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
X_test_scaled

In [ ]:
from sklearn.linear_model import LinearRegression
regressor = LinearRegression()

In [ ]:
regressor.fit(X_train_scaled, y_train)

In [ ]:
regressor.get_params()

In [ ]:
regressor.coef_

In [ ]:
regressor.intercept_

In [ ]:
# y = mx+b
m = regressor.coef_[0][0]
b = regressor.intercept_[0]

In [ ]:
m,b

In [ ]:
x_train_predict = regressor.predict(X_train_scaled)

In [ ]:
plt.scatter(X_train,y_train)
plt.plot(X_train, x_train_predict, color='r')
plt.xlabel("USD_INR")
plt.ylabel("Goldrate")

plt.show()

In [ ]:
X_test_predicted = regressor.predict(X_test_scaled)

In [ ]:
X_test_predicted

In [ ]:
y_test

In [ ]:
from sklearn.metrics import mean_squared_error

In [ ]:
mean_squared_error(y_test, X_test_predicted)

## Hyperparameter optimization

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
param_space = {'copy_X': [True,False], 
               'fit_intercept': [True,False], 
               'n_jobs': [1,5,10,15,None], 
               'positive': [True,False]}

In [ ]:
search = RandomizedSearchCV(regressor, param_space, n_iter=50, cv=5)

In [ ]:
search.fit(X_train_scaled, y_train)

In [ ]:
search.best_params_

In [ ]:
tuned_model = LinearRegression(positive= True, n_jobs= 1, fit_intercept= True, copy_X= True)

In [ ]:
tuned_model.fit(X_train_scaled, y_train)

In [ ]:
tuned_model.coef_

In [ ]:
tuned_model.intercept_

In [ ]:
import pickle

In [ ]:
pickle.dump(regressor,open('regressor.pkl','wb'))

In [ ]:
regressor_reloaded=pickle = pickle.load(open('regressor.pkl','rb'))

In [ ]:
regressor_reloaded.coef_

In [ ]:
# pip install streamlit
import streamlit as st
import pickle
import numpy as np

# Load model
model = pickle.load(open("regressor.pkl", "rb"))
scaler = pickle.load(open("scaler.pkl", "rb"))

# Title
st.title("Gold Price Prediction App")

st.write("Predict Gold Price using USD/INR Rate")

# User input
usd_rate = st.number_input("Enter USD/INR Rate", min_value=50.0, max_value=120.0)

# Predict button
if st.button("Predict Gold Price"):
    
    # Scale input
    usd_scaled = scaler.transform([[usd_rate]])
    
    # Prediction
    prediction = model.predict(usd_scaled)
    
    gold_price_1g = prediction[0][0] / 10
    st.success(f"Predicted Gold Price: ₹ {gold_price_1g:.2f} per gram")

In [ ]:
import pandas as pd

gold_dataset= pd.read_csv("gold_weekly_dataset.csv")

In [ ]:
print(gold_dataset.columns)

In [ ]:
gold_dataset["gold_price_scaled"] = gold_dataset["Gold_Price_per_Gram_INR"] / 10

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import pickle

# Load dataset
gold_dataset = pd.read_csv("gold_weekly_dataset.csv")

# Convert gold price from 10g to 1g
gold_dataset["gold_price_scaled"] = gold_dataset["Gold_Price_per_Gram_INR"] / 10

# Features and target
X = gold_dataset[["USD_Rate"]]
y = gold_dataset["gold_price_scaled"]

# Scale data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train model
model = LinearRegression()
model.fit(X_scaled, y)

# Save model
pickle.dump(model, open("regressor.pkl","wb"))
pickle.dump(scaler, open("scaler.pkl","wb"))

In [ ]:
import streamlit as st

st.title("Gold Price Prediction App")

usd_rate = st.number_input("Enter USD Rate")

st.write("USD Rate entered:", usd_rate)